# Modelo y Evaluación

# Modelo

## El modelo que vamos a aplicar es una red neuronal que aprende con el descenso del gradiente

En esta parte vamos a explicar las matematicas aplicadas a el programa que vamos a crear para que sea entendible

---

## Cálculo de la función de coste

Para clasificación binaria, la función de coste más utilizada es la **entropía cruzada**:

$ J(\theta) = -\frac{1}{n} \sum_{i=1}^{n} \left[ y_i \log(\hat{y}_i) + (1 - y_i)\log(1 - \hat{y}_i) \right] $

donde:

* $y_i \in {0,1}$ es la etiqueta real
* $\hat{y}_i = \sigma(z_i)$ es la predicción de la red (σ es la función sigmoide)
* $z_i = w^T x_i + b$ es la combinación lineal de entradas y pesos

---

## Cálculo del descenso del gradiente

El descenso del gradiente ajusta los parámetros en la dirección opuesta al gradiente de la función de coste:

$ \theta := \theta - \alpha \nabla_{\theta} J(\theta) $

donde $ \alpha $ es la tasa de aprendizaje y $ \nabla_{\theta} J(\theta) $ el gradiente respecto a los pesos y sesgos.

---

## Forward pass para capas

Para una red con (L) capas:

1. **Capa 1 a L-1 (capas ocultas)**:
   $$
   z^{[l]} = W^{[l]} a^{[l-1]} + b^{[l]}, \quad a^{[l]} = \sigma(z^{[l]}), \quad l=1,\dots,L-1
   $$

2. **Capa de salida (capa L)**:
   $$
   z^{[L]} = W^{[L]} a^{[L-1]} + b^{[L]}, \quad \hat{y} = a^{[L]} = \sigma(z^{[L]})
   $$

---

## Backpropagation para capas

1. **Error en la salida**:
   $$
   \delta^{[L]} = a^{[L]} - y
   $$

2. **Propagación hacia atrás para las capas ocultas** $l = L-1, L-2, \dots, 1$:
   $$
   \delta^{[l]} = (W^{[l+1]})^T \delta^{[l+1]} \odot \sigma'(z^{[l]})
   $$
   donde $\odot$ es multiplicación elemento a elemento y $\sigma'(z^{[l]}) = a^{[l]} (1 - a^{[l]})$

3. **Gradientes para cada capa**:
   $$
   \frac{\partial J}{\partial W^{[l]}} = \delta^{[l]} (a^{[l-1]})^T, \quad
   \frac{\partial J}{\partial b^{[l]}} = \delta^{[l]}
   $$


In [ ]:
import numpy as np
from typing import List, Literal

class Model:
    def __init__(self, hidden_layers: List[int], input_layer: int, 
                 activation_function: Literal['Sigmoid'] = 'Sigmoid', lr: float = 0.01):
        """Modelo para hacer clasificación (Red neuronal básica)

        Args:
            hidden_layers (List[int]): Lista con el número de neuronas por capa oculta
            input_layer (int): Número de neuronas de la capa de entrada
            activation_function (Literal['Sigmoid']): Funcion de activación solo disponible sigmoid
            lr (float): learning rate para el descenso del gradiente
        """

        if activation_function != 'Sigmoid':
            raise ValueError('Solo se admite la función de activación Sigmoid')
        self.activation_function = activation_function
        self.lr = lr
        self.params = {}  # Diccionario para pesos y sesgos
        self.layers = [input_layer] + hidden_layers + [1]  # Capas incluyendo salida

    def sigmoid(self, z):
        """Función de activación Sigmoid"""
        return 1 / (1 + np.exp(-z))

    def sigmoid_derivative(self, a):
        """Derivada de la sigmoid en términos de la activación"""
        return a * (1 - a)

    def initialize_params(self):
        """Inicializa pesos y sesgos para todas las capas"""
        np.random.seed(42)  # para reproducibilidad
        for l in range(1, len(self.layers)):
            self.params[f"W{l}"] = np.random.randn(self.layers[l], self.layers[l-1]) * 0.1
            self.params[f"b{l}"] = np.zeros((self.layers[l], 1))

    def forward(self, X):
        """Propagación hacia adelante"""
        cache = {"A0": X.T}  # Guardamos activaciones para backprop
        A_prev = X.T
        L = len(self.layers)
        for l in range(1, L):
            W = self.params[f"W{l}"]
            b = self.params[f"b{l}"]
            Z = W @ A_prev + b  # Z = W*A_prev + b
            A = self.sigmoid(Z)  # Activación
            cache[f"Z{l}"] = Z
            cache[f"A{l}"] = A
            A_prev = A
        return A, cache

    def compute_gradients(self, X, y, cache):
        """Calcula gradientes usando backpropagation"""
        grads = {}
        m = X.shape[0]
        L = len(self.layers)
        y = y.reshape(1, -1)  # Asegurar que y sea (1, m)
        # Error en la capa de salida
        dA = cache[f"A{L}"] - y
        for l in reversed(range(1, L)):
            A_prev = cache[f"A{l-1}"]
            Z = cache[f"Z{l}"]
            dZ = dA * self.sigmoid_derivative(cache[f"A{l}"])
            grads[f"dW{l}"] = (1/m) * dZ @ A_prev.T
            grads[f"db{l}"] = (1/m) * np.sum(dZ, axis=1, keepdims=True)
            if l > 1:
                W = self.params[f"W{l}"]
                dA = W.T @ dZ
        return grads

    def update_params(self, grads):
        """Actualiza los pesos y sesgos con el gradiente descendente"""
        L = len(self.layers)
        for l in range(1, L):
            self.params[f"W{l}"] -= self.lr * grads[f"dW{l}"]
            self.params[f"b{l}"] -= self.lr * grads[f"db{l}"]

    def learn(self, X, y, epochs=1000):
        """Entrenamiento de la red neuronal"""
        self.initialize_params()
        for epoch in range(epochs):
            # Forward pass
            A_out, cache = self.forward(X)
            # Gradientes
            grads = self.compute_gradients(X, y, cache)
            # Actualización de parámetros
            self.update_params(grads)
            if epoch % 100 == 0:
                loss = -np.mean(y*np.log(A_out.T + 1e-8) + (1-y)*np.log(1-A_out.T + 1e-8))
                print(f"Epoch {epoch}, Loss: {loss:.4f}")

    def predict(self, X, desition_barrier: float) -> int:
        """Realiza predicciones binarias

        Args:
            X (_type_): input para predecir
            desition_barrier (float): barrera de desición

        Returns:
            int: devuelve un int con la categoria de la respuesta [0, 1]
        """
        A_out, _ = self.forward(X)
        return (A_out > desition_barrier).astype(int).T